# Ladle life: time-series model development

This notebook uses date, shift, and heat number only to order each ladle's history. The model inputs are ladle number, the five process measurements, and past-only campaign features.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from pipeline.clean import load_and_clean
from pipeline.features import FEATURE_COLUMNS, create_training_features
from pipeline.train import train_and_save


## Clean and inspect the source data
The raw workbook is never changed. Cleaned/model-ready output is written to `data/processed`.

In [2]:
raw_path = ROOT / 'data' / 'raw' / 'ladle_data_synthetic.xlsx'
cleaned = load_and_clean(raw_path)
cleaned[['ladle_no', 'date', 'heat_no', 'ladle_life', 'opening_temp', 'despatch_temp', 'arcing_time', 'purging_time', 'treatment_time']].head()

,ladle_no,date,heat_no,ladle_life,opening_temp,despatch_temp,arcing_time,purging_time,treatment_time
0,8,2025-06-01,26160001,1.0,1570.0,1579.0,27.6,11.3,24.8
1,8,2025-06-01,26160002,2.0,1565.0,1590.0,25.8,21.5,42.4
2,8,2025-06-01,26160022,5.0,1550.0,1576.0,34.9,11.8,18.0
3,8,2025-06-01,26160040,6.0,1589.0,1595.0,20.3,18.9,32.0
4,8,2025-06-01,26160041,7.0,1552.0,1580.0,14.9,20.0,38.2


## Build leakage-safe cycle features
Cumulative, lag, and rolling values look only at previous heats within the same identified ladle campaign.

In [3]:
featured = create_training_features(cleaned)
print(f'{len(FEATURE_COLUMNS)} model features')
featured[FEATURE_COLUMNS + ['ladle_life', 'campaign_id']].head()

27 model features


,ladle_no,opening_temp,despatch_temp,arcing_time,purging_time,treatment_time,temp_gain,total_process_time,campaign_heats_observed,prior_unobserved_heats,...,roll3_arcing_time,roll3_purging_time,roll3_treatment_time,roll3_temp_gain,roll5_arcing_time,roll5_purging_time,roll5_treatment_time,roll5_temp_gain,ladle_life,campaign_id
0,8,1570.0,1579.0,27.6,11.3,24.8,9.0,63.7,1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1
1,8,1565.0,1590.0,25.8,21.5,42.4,25.0,89.7,2,0.0,...,27.600000,11.300000,24.8,9.0,27.600000,11.300000,24.8,9.0,2.0,1
2,8,1550.0,1576.0,34.9,11.8,18.0,26.0,64.7,3,0.0,...,26.700000,16.400000,33.6,17.0,26.700000,16.400000,33.6,17.0,5.0,1
3,8,1589.0,1595.0,20.3,18.9,32.0,6.0,71.2,4,0.0,...,29.433333,14.866667,28.4,20.0,29.433333,14.866667,28.4,20.0,6.0,1
4,8,1552.0,1580.0,14.9,20.0,38.2,28.0,73.1,5,0.0,...,27.000000,17.400000,30.8,19.0,27.150000,15.875000,29.3,16.5,7.0,1


## Train and select the production model
The newest 20% of dates are reserved for testing; records are not randomly shuffled.

In [4]:
metadata = train_and_save(raw_path)
pd.DataFrame(metadata['metrics']).T

,mae,rmse,r2
extra_trees,19.3153,22.8129,-0.9590
random_forest,19.4580,23.0119,-0.9933


Saved outputs: `data/processed/ladle_features.csv`, `src/models/ladle_life_pipeline.pkl`, and `src/models/model_metadata.json`. Start the API only after this cell completes.